In [1]:
import ollama

response = ollama.chat(
    model="mistral",
    messages=[
        {"role": "user", "content": "Say hello in one sentence."}
    ]
)

print(response['message']['content'])

 Hello there! How can I assist you today?


In [2]:
sample_text = """
Customer reported battery overheating issue in LG Smart TV.
Suggested resolution is factory reset.
"""

prompt = f"""
You are an AI system that extracts knowledge graph triples.

Extract triples strictly in this format:
(Entity, Relation, Entity)

Text:
{sample_text}

Only output triples. No explanations.
"""

response = ollama.chat(
    model="mistral",
    messages=[{"role": "user", "content": prompt}]
)

print(response['message']['content'])

 1. (Customer, reports, Battery_Overheating_Issue)
2. (LG_Smart_TV, has, Battery_Overheating_Issue)
3. (Customer, receives_suggestion, Factory_Reset)
4. (LG_Smart_TV, undergoes_operation, Factory_Reset)


In [3]:
import re

raw_output = response['message']['content']

triples = re.findall(r"\((.*?)\)", raw_output)

parsed_triples = []

for triple in triples:
    parts = [part.strip() for part in triple.split(",")]
    if len(parts) == 3:
        parsed_triples.append(parts)

parsed_triples

[['Customer', 'reports', 'Battery_Overheating_Issue'],
 ['LG_Smart_TV', 'has', 'Battery_Overheating_Issue'],
 ['Customer', 'receives_suggestion', 'Factory_Reset'],
 ['LG_Smart_TV', 'undergoes_operation', 'Factory_Reset']]

In [13]:
import pandas as pd
import ollama
import re

# Load dataset
df = pd.read_excel("../data/processed/cleaned_tickets.xlsx")

llm_triples = []

for index, row in df.head(20).iterrows():
    
    text = row['Ticket Description']
    
    product = row['Product Purchased']

    prompt = f"""
The product involved is: {product}

From the text below, identify:

1. What specific problem the product is experiencing.
2. What action is required to fix it.

Return only valid triples in this format:
({product}, EXPERIENCING, Specific problem)
({product}, REQUIRED_ACTION, Specific action)

Do not create new entities.
Do not use 'User'.
Do not use placeholders.
If unclear, return nothing.

Text:
{text}
"""

    response = ollama.chat(
        model="mistral",
        messages=[{"role": "user", "content": prompt}]
    )

    raw_output = response['message']['content']

    triples = re.findall(r"\((.*?)\)", raw_output)

    for triple in triples:
        parts = [part.strip() for part in triple.split(",")]

        if len(parts) == 3:
            subject, relation, obj = parts

            # Remove accidental labels
            subject = subject.replace("Product Name:", "").strip()
            relation = relation.replace("RELATION:", "").strip()
            obj = obj.replace("SPECIFIC DETAIL:", "").strip()

            # Filter weak outputs
            if obj.lower() in ["unknown", "unspecified", "issue"]:
                continue

            llm_triples.append([subject, relation, obj])

llm_triples

[['GoPro Hero',
  'EXPERIENCING',
  'Issue persists after troubleshooting steps mentioned in the user manual'],
 ['GoPro Hero', 'REQUIRED_ACTION', 'Contact support for further assistance'],
 ['LG Smart TV', 'EXPERIENCING', 'Intermittent issues'],
 ['LG Smart TV', 'REQUIRED_ACTION', 'Assistance required'],
 ['Dell XPS', 'EXPERIENCING', 'Not turning on'],
 ['Dell XPS', 'REQUIRED_ACTION', 'Troubleshoot power issues'],
 ['Microsoft Office', 'EXPERIENCING', 'Unresolved issue'],
 ['Microsoft Office',
  'REQUIRED_ACTION',
  'Check Feedback or contact customer support again'],
 ['Autodesk AutoCAD', 'EXPERIENCING', 'Sudden decrease in battery life'],
 ['Autodesk AutoCAD',
  'REQUIRED_ACTION',
  'Seek assistance to resolve the battery life issue'],
 ['Microsoft Office', 'EXPERIENCING', 'Not turning on'],
 ['Microsoft Office',
  'REQUIRED_ACTION',
  'Troubleshooting or contacting technical support'],
 ['Microsoft Surface', 'EXPERIENCING', 'Unable to access account'],
 ['Microsoft Surface', 'REQUI

In [14]:
llm_triples_df = pd.DataFrame(llm_triples, columns=["Subject", "Predicate", "Object"])

llm_triples_df.head()

,Subject,Predicate,Object
0,GoPro Hero,EXPERIENCING,Issue persists after troubleshooting steps men...
1,GoPro Hero,REQUIRED_ACTION,Contact support for further assistance
2,LG Smart TV,EXPERIENCING,Intermittent issues
3,LG Smart TV,REQUIRED_ACTION,Assistance required
4,Dell XPS,EXPERIENCING,Not turning on


In [15]:
llm_triples_df.to_csv("../data/processed/llm_triples.csv", index=False)

print("LLM triples saved successfully!")

LLM triples saved successfully!


In [16]:
structured_df = pd.read_csv("../data/processed/structured_triples.csv")

final_triples_df = pd.concat([structured_df, llm_triples_df], ignore_index=True)

final_triples_df.to_csv("../data/processed/final_triples.csv", index=False)

print("Final merged triples saved successfully!")

Final merged triples saved successfully!


In [1]:
from neo4j import GraphDatabase

URI = "bolt://localhost:7687"
USERNAME = "neo4j"
PASSWORD = "Satwik786"

driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))

# Test connection
with driver.session() as session:
    result = session.run("RETURN 'Connection Successful!' AS message")
    print(result.single()["message"])

driver.close()

Connection Successful!


In [ ]:
from neo4j import GraphDatabase
import pandas as pd

# Neo4j connection
URI = "bolt://localhost:7687"
USERNAME = "neo4j"
PASSWORD = "Satwik786"

driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))

# Load triples
triples_df = pd.read_csv("../data/processed/final_triples.csv")

def create_graph(tx, subject, predicate, obj):
    query = f"""
    MERGE (a:Entity {{name: $subject}})
    MERGE (b:Entity {{name: $object}})
    MERGE (a)-[r:{predicate}]->(b)
    """
    tx.run(query, subject=subject, object=obj)

with driver.session() as session:
    for index, row in triples_df.iterrows():
        session.execute_write(create_graph, row["Subject"], row["Predicate"], row["Object"])

driver.close()

print("Graph construction completed!")